# Download Duck land elevation data and combine into one .nc-file

In [1]:
download_data = False
merge_data = False

## Download data
Code produced mainly by Duck AI chat with GPT-4o

In [5]:
path = '/home/jovyan/data/98_paper2_kf_dtm/02_Duck_validation_data/'

In [2]:
import requests
import xml.etree.ElementTree as ET
import os

import xarray as xr

import numpy as np

In [3]:
if download_data:
    # Find existing files
    existing_files = []
    for root, dirs, files in os.walk(path):
        for fname in files:
            if fname.endswith('.nc'):
                existing_files.append(fname)

    # Find all files in xml catalog
    all_dataset_names = []

    # URL of the THREDDS XML catalog
    thredds_catalog_url = 'https://chldata.erdc.dren.mil/thredds/catalog/frf/geomorphology/DEMs/surveyDEM/data/catalog.xml'

    # Send a GET request to fetch the XML catalog
    response = requests.get(thredds_catalog_url)

    # Check if the request was successful
    if response.status_code == 200:
        # Parse the XML content
        root = ET.fromstring(response.content)
        # namespace
        ns = {'thredds': 'http://www.unidata.ucar.edu/namespaces/thredds/InvCatalog/v1.0'}

        # Find all datasets
        for dataset in root.findall('.//thredds:dataset', ns):
            all_dataset_names.append(dataset.get('name'))
    else:
        print("Failed to fetch XML catalog:", response.status_code)

    # Download the files from the server, that are not existing yet in the folder
    thredds_server = 'https://chldata.erdc.dren.mil/thredds/fileServer/frf/geomorphology/DEMs/surveyDEM/data/'
    for dataset_name in all_dataset_names[1:]:
        if dataset_name in existing_files:
            continue
        else:
            full_dataset_url = f"{thredds_server}/{dataset_name}"
            print(f"Downloading {dataset_name} from {full_dataset_url}...")
            file_response = requests.get(full_dataset_url)
            if file_response.status_code == 200:
                # Define the local file path
                local_file_path = os.path.join(path, dataset_name)

                # Save the file
                with open(local_file_path, 'wb') as f:
                    f.write(file_response.content)
                print(f"Downloaded {dataset_name} successfully to {local_file_path}.")
            else:
                print(f"Failed to download {full_dataset_url}: {file_response.status_code}")

## Merge all .nc-files

In [6]:
fn = 'duck_validation.nc'
if (not os.path.isfile(path+fn)) | merge_data:
    all_data = xr.Dataset()

    for root, dirs, files in os.walk(path):        
        for i, fname in enumerate(files):
            if fname.endswith('.nc'):
                try:
                    data_temp = xr.open_dataset(os.path.join(root, fname))
                    if i > 0:
                    # keep the first latitude/longitude
                    # delete the other ones to avoid merging error (see comment below)
                        data_temp = data_temp.drop_vars(['latitude', 'longitude'])
                except:
                    print(f'Failed loading {fname}')
                try:
                    all_data = all_data.merge(data_temp)
                except:
                    print(f'Failed to merge {fname}')

    all_data.to_netcdf(path+fn)
else:
    all_data = xr.open_dataset(path+fn)

From end of 2019 on, longitude and latitudes are about 1e-6 different from before (x and y are the same), which translates to a few decimeters difference. Seeting compat='override' means that also elevation is overwritten, therefore dropping variable after the first dataset.  
  
There were daily observations in 09/1997 so will not try to figure out why this file is not loading.

In [8]:
all_data

<xarray.Dataset> Size: 15MB
Dimensions:                (time: 492, xFRF: 76, yFRF: 51, project: 16)
Coordinates:
  * time                   (time) datetime64[ns] 4kB 1981-07-02 ... 2025-04-15
  * xFRF                   (xFRF) float64 608B 50.0 62.0 74.0 ... 938.0 950.0
  * yFRF                   (yFRF) float64 408B -100.0 -76.0 ... 1.1e+03
    project                (time, project) |S1 8kB ...
Data variables:
    elevation              (time, yFRF, xFRF) float64 15MB ...
    latitude               (yFRF, xFRF) float64 31kB ...
    longitude              (yFRF, xFRF) float64 31kB ...
    surveyNumber           (time) float64 4kB ...
    surveyVehicle          (time) float32 2kB ...
    versionDate            (time) datetime64[ns] 4kB ...
    surveyInstrumentation  (time) float32 2kB ...

In [10]:
# double-check epsg for an early survey
xr.open_dataset(path + 'FRF_geomorphology_DEMs_surveyDEM_19810702.nc')

<xarray.Dataset> Size: 94kB
Dimensions:        (time: 1, xFRF: 76, yFRF: 51, project: 16)
Coordinates:
  * time           (time) datetime64[ns] 8B 1981-07-02
  * xFRF           (xFRF) float64 608B 50.0 62.0 74.0 86.0 ... 926.0 938.0 950.0
  * yFRF           (yFRF) float64 408B -100.0 -76.0 -52.0 ... 1.076e+03 1.1e+03
    project        (time, project) object 128B ...
Data variables:
    elevation      (time, yFRF, xFRF) float64 31kB ...
    latitude       (yFRF, xFRF) float64 31kB ...
    longitude      (yFRF, xFRF) float64 31kB ...
    surveyNumber   (time) float64 8B ...
    surveyVehicle  (time) float32 4B ...
    versionDate    (time) datetime64[ns] 8B ...
Attributes: (12/46)
    featureType:                     bathymetry grid
    title:                           bathymetry grid
    summary:                         USACE/COAB transect made to a grided file\n
    history:                         stuff
    source:                          stuff
    sourceUrl:                       (local files)
    ...                              ...
    geospatial_vertical_positive:    up
    cross_shore_angle_units:         degrees_north
    cross_shore_angle_description:   cross shore angle at the USACE FRF site ...
    cross_shore_angle:               71.8
    date_created:                    2020-10-31
    date_issued:                     2020-10-31

In [14]:
# cross-shore spacing
np.diff(all_data.xFRF)

array([12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12.,
       12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12.,
       12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12.,
       12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12.,
       12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12.,
       12., 12., 12., 12., 12., 12., 12., 12., 12., 12.])

In [17]:
# alongshore spacing
np.diff(all_data.yFRF)

array([24., 24., 24., 24., 24., 24., 24., 24., 24., 24., 24., 24., 24.,
       24., 24., 24., 24., 24., 24., 24., 24., 24., 24., 24., 24., 24.,
       24., 24., 24., 24., 24., 24., 24., 24., 24., 24., 24., 24., 24.,
       24., 24., 24., 24., 24., 24., 24., 24., 24., 24., 24.])

In [21]:
all_data.yFRF

<xarray.DataArray 'yFRF' (yFRF: 51)> Size: 408B
array([-100.,  -76.,  -52.,  -28.,   -4.,   20.,   44.,   68.,   92.,  116.,
        140.,  164.,  188.,  212.,  236.,  260.,  284.,  308.,  332.,  356.,
        380.,  404.,  428.,  452.,  476.,  500.,  524.,  548.,  572.,  596.,
        620.,  644.,  668.,  692.,  716.,  740.,  764.,  788.,  812.,  836.,
        860.,  884.,  908.,  932.,  956.,  980., 1004., 1028., 1052., 1076.,
       1100.])
Coordinates:
  * yFRF     (yFRF) float64 408B -100.0 -76.0 -52.0 ... 1.076e+03 1.1e+03
Attributes:
    units:       m
    long_name:   y-coordinate in Local FRF Cartesian system
    data_type:   f8
    dim:         yFRF
    fill_value:  -999
    short_name:  y
    notes:       this is the alongshore coordinate in the local FRF coordinat...
    axis:        Y